> **Save Your Work** — Before you begin, click **File → Save a copy in Drive** (Google Colab) or **File → Download** so you do not lose your progress.

# Module 7 Assessment — Introduction to Machine Learning

Build a complete data preparation pipeline on the Palmer Penguins dataset.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

df = sns.load_dataset("penguins")
print(df.shape)
print(df.dtypes)
df.head()
# (344, 7)
# species       object
# island        object
# bill_length_mm float64
# bill_depth_mm  float64
# flipper_length_mm float64
# body_mass_g   float64
# sex           object

## Task 1: Identify the ML Problem Type

In the markdown cell below, answer:
1. Is this supervised or unsupervised learning? Why?
2. Is this classification or regression? Why?
3. Is this binary or multi-class classification? Why?

1. **Supervised learning** — each row has a known label (`species`), so we can train a model that maps features to a target with ground truth.
2. **Classification** — the target (`species`) is a discrete category, not a continuous numeric value.
3. **Multi-class classification** — there are three species (Adelie, Chinstrap, Gentoo), which is more than two classes.

## Task 2: Inspect and Clean the Data

Perform the following:
1. Print shape, dtypes, and missing value counts
2. Drop any duplicate rows
3. Impute missing values: numeric → median, `sex` → mode
4. Confirm zero missing values with `isnull().sum()`

In [ ]:
# 1. Inspect
print("Shape:", df.shape)
print("\nDtypes:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
# bill_length_mm      2
# bill_depth_mm       2
# flipper_length_mm   2
# body_mass_g         2
# sex                11

# 2. Drop duplicates
df.drop_duplicates(inplace=True)
print("\nAfter dropping duplicates:", df.shape)

# 3. Impute numeric columns with median
numeric_cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
for col in numeric_cols:
    df[col].fillna(df[col].median(), inplace=True)

# Impute sex with mode
df["sex"].fillna(df["sex"].mode()[0], inplace=True)

# 4. Confirm zero missing values
print("\nMissing values after imputation:")
print(df.isnull().sum())
# All zeros

## Task 3: Define Features and Target

1. Set `y = df['species']`, `X = df.drop(columns=['species'])`
2. Stratified train/test split: 80/20, `random_state=42`
3. Print shapes and class distribution in `y_train`

In [ ]:
from sklearn.model_selection import train_test_split

y = df["species"]
X = df.drop(columns=["species"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train shape:", X_train.shape)   # (275, 6)
print("X_test shape: ", X_test.shape)    # (69, 6)
print("y_train shape:", y_train.shape)   # (275,)
print("y_test shape: ", y_test.shape)    # (69,)
print("\nClass distribution in y_train:")
print(y_train.value_counts())
# Adelie      120
# Gentoo       96
# Chinstrap    59

## Task 4: Engineer a New Feature

Create `bill_ratio = bill_length_mm / bill_depth_mm` on both train and test sets (after splitting).

In [ ]:
X_train = X_train.copy()
X_test  = X_test.copy()

X_train["bill_ratio"] = X_train["bill_length_mm"] / X_train["bill_depth_mm"]
X_test["bill_ratio"]  = X_test["bill_length_mm"]  / X_test["bill_depth_mm"]

print("bill_ratio sample values:")
print(X_train["bill_ratio"].describe())
# Gentoo penguins tend to have higher bill_ratio (~2.3)
# Chinstrap penguins tend to have lower bill_ratio (~2.0)

## Task 5: Build a ColumnTransformer Pipeline

Build a preprocessor:
- **Numeric** (`bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`, `bill_ratio`): `StandardScaler`
- **Categorical** (`island`, `sex`): `OneHotEncoder(drop='first', sparse_output=False)`

Fit on training data only. Transform both sets.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_features     = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g", "bill_ratio"]
categorical_features = ["island", "sex"]

numeric_transformer     = StandardScaler()
categorical_transformer = OneHotEncoder(drop="first", sparse_output=False)

preprocessor = ColumnTransformer([
    ("num", numeric_transformer,     numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# Fit on training data only — never fit on test data
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed  = preprocessor.transform(X_test)

print("X_train_processed shape:", X_train_processed.shape)
print("X_test_processed shape: ", X_test_processed.shape)
# 5 numeric + 2 island dummies + 1 sex dummy = 8 features

## Task 6: Verify the Output

1. Confirm no NaN values in `X_train_processed`
2. Check that scaled numeric columns have approximately mean=0, std=1
3. Print total number of output features and what they represent

In [ ]:
# 1. Check for NaN values
nan_count = np.isnan(X_train_processed).sum()
print("NaN count in X_train_processed:", nan_count)  # 0

# 2. Check mean and std of numeric columns (first 5 columns)
numeric_processed = X_train_processed[:, :5]
print("\nNumeric column means (should be ~0):")
print(numeric_processed.mean(axis=0).round(6))
print("Numeric column stds (should be ~1):")
print(numeric_processed.std(axis=0).round(6))

# 3. Feature names
cat_feature_names = preprocessor.named_transformers_["cat"].get_feature_names_out(categorical_features)
all_feature_names = numeric_features + list(cat_feature_names)
print("\nTotal output features:", X_train_processed.shape[1])
print("Feature names:", all_feature_names)
# ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'bill_ratio',
#  'island_Dream', 'island_Torgersen', 'sex_Male']

## Task 7: Reflection

In the cell below, answer:
1. Why must the preprocessor be **fitted on training data only**?
2. Propose one additional engineered feature and explain its signal.
3. If the model achieves 95% accuracy but Chinstrap penguins are 20% of data — is accuracy alone sufficient?

1. **Fitting on training data only** prevents data leakage. If we fit the scaler on the full dataset (including test data), the scaler learns the test set's mean and standard deviation. During evaluation, the model would have indirectly "seen" test data statistics, producing an overly optimistic performance estimate that does not reflect true generalization.

2. **`flipper_body_ratio = flipper_length_mm / body_mass_g`** could capture body proportion differences across species. Gentoo penguins are heavier but have proportionally long flippers relative to smaller Adelie penguins, so this ratio may add discriminative signal beyond the raw measurements alone.

3. Accuracy alone is **not sufficient** when classes are imbalanced. If the model always predicts Adelie (the majority class), it could achieve ~80% accuracy while completely failing on Chinstrap penguins. Per-class precision, recall, and F1 scores — or a macro-averaged F1 — give a more honest picture of performance across all three species.